# 파이썬 머신러닝 완벽 가이드 4장

분류 실습: 캐글 산탄데르 고객 만족 예측, 캐글 신용카드 사기 검출 (p.267~295)

## 실습 환경 설정

과제 조건에 따라 XGBoost 1.7.6과 LightGBM 4.1.0을 설치한다. 설치 후에는 **런타임 → 세션 다시 시작**을 한 번 실행한 뒤 아래 셀부터 진행한다.

- LightGBM 4.x부터는 `fit()`에 `early_stopping_rounds`, `verbose` 인자를 넣을 수 없다. 책 코드의 조기 중단은 `callbacks=[early_stopping(...)]`로 바꿔 적는다.
- XGBoost 1.7.6은 책 코드처럼 `fit()`에 `early_stopping_rounds`, `eval_metric`을 넣어도 동작한다(경고만 출력된다).

In [ ]:
!pip install xgboost==1.7.6
!pip install lightgbm==4.1.0
!pip install hyperopt

In [ ]:
import sklearn
import xgboost
import lightgbm

print(sklearn.__version__, xgboost.__version__, lightgbm.__version__)

## 09. 분류 실습 - 캐글 산탄데르 고객 만족 예측

### 개념 정리

**문제 정의**
- 산탄데르 은행이 캐글에 공개한 데이터로, 370개 피처를 이용해 고객 만족 여부를 예측한다.
- 피처 이름은 모두 익명화되어 있어 의미를 알 수 없다.
- 레이블 `TARGET`이 **1이면 불만족, 0이면 만족** 고객이다.

**평가 지표: ROC-AUC**
- 대부분이 만족 고객이고 불만족 고객은 약 4%에 불과한 **불균형 데이터**이다.
- 모두 만족(0)으로 예측해도 정확도가 96%가 나오므로 정확도는 의미가 없다.
- 그래서 ROC 곡선 아래 면적인 **ROC-AUC**로 성능을 평가한다.

**실습 흐름**
1. 데이터 전처리(이상값 대체, 불필요한 ID 컬럼 삭제)
2. 학습/테스트 분할 → 학습 데이터를 다시 학습/검증으로 분할(조기 중단용)
3. XGBoost 기본 학습 → HyperOpt 튜닝 → 재학습
4. LightGBM 기본 학습 → HyperOpt 튜닝 → 재학습

데이터는 캐글 Santander Customer Satisfaction의 `train.csv`를 내려받아 `train_santander.csv`로 이름을 바꾼 뒤 Colab `/content`에 업로드한다.

### 데이터 전처리

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib

cust_df = pd.read_csv('/content/train_santander.csv', encoding='latin-1')
print('dataset shape:', cust_df.shape)
cust_df.head(3)

In [ ]:
cust_df.info()

- 111개 피처가 float64, 260개 피처가 int64로 **모두 숫자형**이고 Null 값도 없다.
- 레이블 분포를 확인해 불만족 고객의 비율을 본다.

In [ ]:
print(cust_df['TARGET'].value_counts())
unsatisfied_cnt = cust_df[cust_df['TARGET'] == 1].TARGET.count()
total_cnt = cust_df.TARGET.count()
print('unsatisfied 비율은 {0:.2f}'.format((unsatisfied_cnt / total_cnt)))

In [ ]:
cust_df.describe()

- `var3` 컬럼의 최솟값이 **-999999**로, NaN이나 특정 예외값을 대신 넣은 것으로 보인다.
- 해당 값은 116개뿐이므로 `var3`에서 가장 많이 나오는 값인 **2로 대체**한다.
- `ID`는 단순 식별자라 예측에 도움이 되지 않으므로 삭제한다.

In [ ]:
print(cust_df.var3.value_counts()[:10])

In [ ]:
# var3 피처 값 대체 및 ID 피처 드롭
cust_df['var3'].replace(-999999, 2, inplace=True)
cust_df.drop('ID', axis=1, inplace=True)

# 피처 세트와 레이블 세트분리. 레이블 컬럼은 DataFrame의 맨 마지막에 위치해 컬럼 위치 -1로 분리
X_features = cust_df.iloc[:, :-1]
y_labels = cust_df.iloc[:, -1]
print('피처 데이터 shape:{0}'.format(X_features.shape))

학습 80%, 테스트 20%로 나눈 뒤, 원본 데이터와 비슷한 비율로 불만족 레이블이 나뉘었는지 확인한다.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X_features, y_labels,
                                                    test_size=0.2, random_state=0)
train_cnt = y_train.count()
test_cnt = y_test.count()
print('학습 세트 Shape:{0}, 테스트 세트 Shape:{1}'.format(X_train.shape , X_test.shape))

print(' 학습 세트 레이블 값 분포 비율')
print(y_train.value_counts()/train_cnt)
print('\n 테스트 세트 레이블 값 분포 비율')
print(y_test.value_counts()/test_cnt)

- 학습·테스트 모두 불만족(1) 비율이 원본과 같은 약 4%로 분할되었다.
- XGBoost와 LightGBM은 **조기 중단** 시 검증 데이터가 필요하다. 테스트 데이터를 검증에 쓰면 테스트 데이터에 과적합될 수 있으므로, 학습 데이터를 다시 **학습 70% / 검증 30%**로 나눈다.

In [ ]:
# X_train, y_train을 다시 학습과 검증 데이터 세트로 분리.
X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train,
                                            test_size=0.3, random_state=0)

### XGBoost 모델 학습과 하이퍼 파라미터 튜닝

먼저 기본 파라미터에 가깝게 학습한다. `n_estimators=500`, `learning_rate=0.05`, 평가 지표는 `auc`, 조기 중단은 100회로 설정한다.

In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score

# n_estimators는 500으로, learning_rate 0.05, random state는 예제 수행 시마다 동일 예측 결과를 위해 설정.
xgb_clf = XGBClassifier(n_estimators=500, learning_rate=0.05, random_state=156)

# 성능 평가 지표를 auc로, 조기 중단 파라미터는 100으로 설정하고 학습 수행.
xgb_clf.fit(X_tr, y_tr, early_stopping_rounds=100, eval_metric='auc',
            eval_set=[(X_tr, y_tr), (X_val, y_val)])

xgb_roc_score = roc_auc_score(y_test, xgb_clf.predict_proba(X_test)[:, 1])
print('ROC AUC: {0:.4f}'.format(xgb_roc_score))

**HyperOpt로 튜닝하기**

| 파라미터 | 검색 공간 | 설명 |
|---|---|---|
| `max_depth` | 5~15, 1 간격 (`hp.quniform`) | 트리 최대 깊이 |
| `min_child_weight` | 1~6, 1 간격 (`hp.quniform`) | 자식 노드 가중치 합의 최솟값 |
| `colsample_bytree` | 0.5~0.95 (`hp.uniform`) | 트리 생성 시 샘플링할 피처 비율 |
| `learning_rate` | 0.01~0.2 (`hp.uniform`) | 학습률 |

**목적 함수 작성 시 주의할 점**
- `hp.quniform`은 실수형 값을 반환하므로 `max_depth`, `min_child_weight`는 `int()`로 변환해 넣는다.
- 과적합을 줄이기 위해 **3개 폴드 교차 검증**으로 평균 ROC-AUC를 구한다. 이때 각 폴드에서 조기 중단을 쓰려고 `cross_val_score()` 대신 `KFold`로 직접 나눈다.
- 수행 시간을 줄이려고 튜닝 단계에서는 `n_estimators=100`, 조기 중단 30회로 설정한다.
- HyperOpt의 `fmin()`은 목적 함수의 **최솟값**을 찾으므로, ROC-AUC 평균에 **-1을 곱해** 반환한다.

In [ ]:
from hyperopt import hp

# max_depth는 5에서 15까지 1간격으로, min_child_weight는 1에서 6까지 1간격으로
# colsample_bytree는 0.5에서 0.95사이, learning_rate는 0.01에서 0.2사이 정규 분포된 값으로 검색.
xgb_search_space = {'max_depth': hp.quniform('max_depth', 5, 15, 1),
                    'min_child_weight': hp.quniform('min_child_weight', 1, 6, 1),
                    'colsample_bytree': hp.uniform('colsample_bytree', 0.5, 0.95),
                    'learning_rate': hp.uniform('learning_rate', 0.01, 0.2)
}

In [ ]:
from sklearn.model_selection import KFold
from sklearn.metrics import roc_auc_score

# 목적 함수 설정.
# 추후 fmin()에서 입력된 search_space값으로 XGBClassifier 교차 검증 학습 후 -1* roc_auc 평균 값을 반환.
def objective_func(search_space):
    xgb_clf = XGBClassifier(n_estimators=100, max_depth=int(search_space['max_depth']),
                            min_child_weight=int(search_space['min_child_weight']),
                            colsample_bytree=search_space['colsample_bytree'],
                            learning_rate=search_space['learning_rate']
                           )

    # 3개 k-fold 방식으로 평가된 roc_auc 지표를 담는 list
    roc_auc_list = []

    # 3개 k-fold방식 적용
    kf = KFold(n_splits=3)
    # X_train을 다시 학습과 검증용 데이터로 분리
    for tr_index, val_index in kf.split(X_train):
        # kf.split(X_train)으로 추출된 학습과 검증 index값으로 학습과 검증 데이터 세트 분리
        X_tr, y_tr = X_train.iloc[tr_index], y_train.iloc[tr_index]
        X_val, y_val = X_train.iloc[val_index], y_train.iloc[val_index]
        # early stopping은 30회로 설정하고 추출된 학습과 검증 데이터로 XGBClassifier 학습 수행.
        xgb_clf.fit(X_tr, y_tr, early_stopping_rounds=30, eval_metric='auc',
                    eval_set=[(X_tr, y_tr), (X_val, y_val)])

        # 1로 예측한 확률값 추출후 roc auc 계산하고 평균 roc auc 계산을 위해 list에 결과값 담음.
        score = roc_auc_score(y_val, xgb_clf.predict_proba(X_val)[:, 1])
        roc_auc_list.append(score)

    # 3개 k-fold로 계산된 roc_auc값의 평균값을 반환하되,
    # HyperOpt는 목적함수의 최소값을 위한 입력값을 찾으므로 -1을 곱한 뒤 반환.
    return -1 * np.mean(roc_auc_list)

`fmin()`으로 50회 반복하며 최적 하이퍼 파라미터를 찾는다. 데이터가 커서 수행 시간이 꽤 오래 걸린다.

In [ ]:
from hyperopt import fmin, tpe, Trials

trials = Trials()

# fmin()함수를 호출. max_evals지정된 횟수만큼 반복 후 목적함수의 최소값을 가지는 최적 입력값 추출.
best = fmin(fn=objective_func,
            space=xgb_search_space,
            algo=tpe.suggest,
            max_evals=50, # 최대 반복 횟수를 지정합니다.
            trials=trials, rstate=np.random.default_rng(seed=30))

print('best:', best)

찾은 최적 파라미터로 `n_estimators`를 500으로 늘려 다시 학습하고 테스트 데이터로 ROC-AUC를 측정한다.

In [ ]:
# n_estimators를 500증가 후 최적으로 찾은 하이퍼 파라미터를 기반으로 학습과 예측 수행.
xgb_clf = XGBClassifier(n_estimators=500, learning_rate=round(best['learning_rate'], 5),
                        max_depth=int(best['max_depth']), min_child_weight=int(best['min_child_weight']),
                        colsample_bytree=round(best['colsample_bytree'], 5)
                       )

# evaluation metric을 auc로, early stopping은 100 으로 설정하고 학습 수행.
xgb_clf.fit(X_tr, y_tr, early_stopping_rounds=100,
            eval_metric='auc', eval_set=[(X_tr, y_tr), (X_val, y_val)])

xgb_roc_score = roc_auc_score(y_test, xgb_clf.predict_proba(X_test)[:, 1])
print('ROC AUC: {0:.4f}'.format(xgb_roc_score))

튜닝 후 ROC-AUC가 소폭 향상된다. 피처 중요도 상위 20개를 시각화해 본다.

In [ ]:
from xgboost import plot_importance
import matplotlib.pyplot as plt
%matplotlib inline

fig, ax = plt.subplots(1, 1, figsize=(10, 8))
plot_importance(xgb_clf, ax=ax, max_num_features=20, height=0.4)

### LightGBM 모델 학습과 하이퍼 파라미터 튜닝

XGBoost와 같은 방식으로 먼저 기본 학습을 한다. LightGBM 4.1.0에서는 조기 중단을 `callbacks`로 지정한다.

In [ ]:
from lightgbm import LGBMClassifier, early_stopping

lgbm_clf = LGBMClassifier(n_estimators=500)

eval_set = [(X_tr, y_tr), (X_val, y_val)]
# 책 코드: lgbm_clf.fit(X_tr, y_tr, early_stopping_rounds=100, eval_metric='auc', eval_set=eval_set)
lgbm_clf.fit(X_tr, y_tr, eval_metric='auc', eval_set=eval_set,
             callbacks=[early_stopping(stopping_rounds=100)])

lgbm_roc_score = roc_auc_score(y_test, lgbm_clf.predict_proba(X_test)[:, 1])
print('ROC AUC: {0:.4f}'.format(lgbm_roc_score))

**HyperOpt로 튜닝하기**

| 파라미터 | 검색 공간 | 설명 |
|---|---|---|
| `num_leaves` | 32~64, 1 간격 | 트리 하나가 가질 수 있는 최대 리프 수 |
| `max_depth` | 100~160, 1 간격 | 트리 최대 깊이 |
| `min_child_samples` | 60~100, 1 간격 | 리프 노드가 되기 위한 최소 데이터 수 |
| `subsample` | 0.7~1 | 데이터 샘플링 비율 |
| `learning_rate` | 0.01~0.2 | 학습률 |

목적 함수 구조는 XGBoost와 같다(정수형 변환, 3폴드 교차 검증, -1 곱해서 반환).

In [ ]:
lgbm_search_space = {'num_leaves': hp.quniform('num_leaves', 32, 64, 1),
                     'max_depth': hp.quniform('max_depth', 100, 160, 1),
                     'min_child_samples': hp.quniform('min_child_samples', 60, 100, 1),
                     'subsample': hp.uniform('subsample', 0.7, 1),
                     'learning_rate': hp.uniform('learning_rate', 0.01, 0.2)
                    }

def objective_func(search_space):
    lgbm_clf = LGBMClassifier(n_estimators=100, num_leaves=int(search_space['num_leaves']),
                              max_depth=int(search_space['max_depth']),
                              min_child_samples=int(search_space['min_child_samples']),
                              subsample=search_space['subsample'],
                              learning_rate=search_space['learning_rate'])
    # 3개 k-fold 방식으로 평가된 roc_auc 지표를 담는 list
    roc_auc_list = []

    # 3개 k-fold방식 적용
    kf = KFold(n_splits=3)
    # X_train을 다시 학습과 검증용 데이터로 분리
    for tr_index, val_index in kf.split(X_train):
        # kf.split(X_train)으로 추출된 학습과 검증 index값으로 학습과 검증 데이터 세트 분리
        X_tr, y_tr = X_train.iloc[tr_index], y_train.iloc[tr_index]
        X_val, y_val = X_train.iloc[val_index], y_train.iloc[val_index]

        # early stopping은 30회로 설정하고 추출된 학습과 검증 데이터로 LGBMClassifier 학습 수행.
        lgbm_clf.fit(X_tr, y_tr, eval_metric='auc',
                     eval_set=[(X_tr, y_tr), (X_val, y_val)],
                     callbacks=[early_stopping(stopping_rounds=30)])

        # 1로 예측한 확률값 추출후 roc auc 계산하고 평균 roc auc 계산을 위해 list에 결과값 담음.
        score = roc_auc_score(y_val, lgbm_clf.predict_proba(X_val)[:, 1])
        roc_auc_list.append(score)

    # 3개 k-fold로 계산된 roc_auc값의 평균값을 반환하되,
    # HyperOpt는 목적함수의 최소값을 위한 입력값을 찾으므로 -1을 곱한 뒤 반환.
    return -1 * np.mean(roc_auc_list)

In [ ]:
from hyperopt import fmin, tpe, Trials

trials = Trials()

# fmin()함수를 호출. max_evals지정된 횟수만큼 반복 후 목적함수의 최소값을 가지는 최적 입력값 추출.
best = fmin(fn=objective_func, space=lgbm_search_space, algo=tpe.suggest,
            max_evals=50, # 최대 반복 횟수를 지정합니다.
            trials=trials, rstate=np.random.default_rng(seed=30))

print('best:', best)

In [ ]:
lgbm_clf = LGBMClassifier(n_estimators=500, num_leaves=int(best['num_leaves']),
                          max_depth=int(best['max_depth']),
                          min_child_samples=int(best['min_child_samples']),
                          subsample=round(best['subsample'], 5),
                          learning_rate=round(best['learning_rate'], 5)
                         )

# evaluation metric을 auc로, early stopping은 100 으로 설정하고 학습 수행.
lgbm_clf.fit(X_tr, y_tr, eval_metric='auc', eval_set=[(X_tr, y_tr), (X_val, y_val)],
             callbacks=[early_stopping(stopping_rounds=100)])

lgbm_roc_score = roc_auc_score(y_test, lgbm_clf.predict_proba(X_test)[:, 1])
print('ROC AUC: {0:.4f}'.format(lgbm_roc_score))

- LightGBM도 튜닝 후 XGBoost와 비슷한 수준의 ROC-AUC를 얻는다.
- 성능은 비슷하지만 **LightGBM의 학습 시간이 훨씬 짧다.** 같은 50회 튜닝도 LightGBM 쪽이 빠르게 끝난다.

## 10. 분류 실습 - 캐글 신용카드 사기 검출

### 개념 정리

**문제 정의**
- 2013년 유럽 카드 사용자들의 거래 데이터로, 284,807건 중 사기(`Class=1`)는 **492건(약 0.172%)**뿐인 극도로 불균형한 데이터이다.
- `V1`~`V28`은 개인정보 보호를 위해 PCA로 변환된 피처이고, `Time`(첫 거래로부터 경과 시간), `Amount`(거래 금액)만 원래 의미가 남아 있다.

**언더 샘플링과 오버 샘플링**

레이블이 불균형하면 이상 레이블을 학습할 데이터가 너무 적어서, 모델이 정상 레이블 쪽으로 치우치게 학습되고 이상 데이터 검출 성능이 떨어진다. 이를 해결하기 위해 학습 데이터의 레이블 분포를 맞춘다.

| 방식 | 방법 | 장단점 |
|---|---|---|
| **언더 샘플링** | 많은 쪽(정상) 데이터를 적은 쪽 수준으로 **줄인다** | 정상 데이터를 너무 많이 버려 정상 레이블 학습이 부족해질 수 있다 |
| **오버 샘플링** | 적은 쪽(이상) 데이터를 많은 쪽 수준으로 **늘린다** | 예측 성능상 더 유리해 많이 쓰인다 |

- 동일한 데이터를 단순히 복사하면 과적합이 되므로, 원본 피처 값을 **약간만 변형해** 새 데이터를 만든다.
- 대표 방법이 **SMOTE(Synthetic Minority Over-sampling Technique)** 이다. 적은 쪽 데이터 각각에 대해 **K 최근접 이웃**을 찾고, 그 이웃과의 차이를 일정 비율로 곱해 사이에 새로운 데이터를 만든다.
- 파이썬에서는 `imbalanced-learn` 패키지로 SMOTE를 쓸 수 있다.

**실습 흐름**
1. 아무 가공 없이 로지스틱 회귀·LightGBM 학습
2. `Amount` 피처 분포 변환(표준화 → 로그 변환)
3. 이상치 제거(IQR)
4. SMOTE 오버 샘플링

데이터는 캐글 Credit Card Fraud Detection의 `creditcard.csv`를 Colab `/content`에 업로드한다.

### 데이터 일차 가공 및 모델 학습/예측/평가

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")
%matplotlib inline

card_df = pd.read_csv('/content/creditcard.csv')
card_df.head(3)

- `Time` 피처는 의미가 없으므로 삭제한다.
- 이후 여러 가공을 단계별로 적용하기 위해 원본 DataFrame은 그대로 두고, **복사본을 가공해 반환하는 함수** `get_preprocessed_df()`를 만든다.
- `get_train_test_dataset()`은 가공 후 학습/테스트를 7:3으로 나누는데, `stratify=y_target`으로 **레이블 분포를 동일하게** 유지한다.

In [ ]:
from sklearn.model_selection import train_test_split

# 인자로 입력받은 DataFrame을 복사 한 뒤 Time 컬럼만 삭제하고 복사된 DataFrame 반환
def get_preprocessed_df(df=None):
    df_copy = df.copy()
    df_copy.drop('Time', axis=1, inplace=True)
    return df_copy

In [ ]:
# 사전 데이터 가공 후 학습과 테스트 데이터 세트를 반환하는 함수.
def get_train_test_dataset(df=None):
    # 인자로 입력된 DataFrame의 사전 데이터 가공이 완료된 복사 DataFrame 반환
    df_copy = get_preprocessed_df(df)
    # DataFrame의 맨 마지막 컬럼이 레이블, 나머지는 피처들
    X_features = df_copy.iloc[:, :-1]
    y_target = df_copy.iloc[:, -1]
    # train_test_split( )으로 학습과 테스트 데이터 분할. stratify=y_target으로 Stratified 기반 분할
    X_train, X_test, y_train, y_test = \
    train_test_split(X_features, y_target, test_size=0.3, random_state=0, stratify=y_target)
    # 학습과 테스트 데이터 세트 반환
    return X_train, X_test, y_train, y_test

X_train, X_test, y_train, y_test = get_train_test_dataset(card_df)

In [ ]:
print('학습 데이터 레이블 값 비율')
print(y_train.value_counts()/y_train.shape[0] * 100)
print('테스트 데이터 레이블 값 비율')
print(y_test.value_counts()/y_test.shape[0] * 100)

학습·테스트 모두 사기 비율이 약 0.17%로 원본과 같다. 평가를 위해 3장에서 만든 `get_clf_eval()`에 ROC-AUC를 추가해 정의한다.

In [ ]:
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

def get_clf_eval(y_test, pred=None, pred_proba=None):
    confusion = confusion_matrix(y_test, pred)
    accuracy = accuracy_score(y_test , pred)
    precision = precision_score(y_test , pred)
    recall = recall_score(y_test , pred)
    f1 = f1_score(y_test,pred)
    # ROC-AUC 추가
    roc_auc = roc_auc_score(y_test, pred_proba)
    print('오차 행렬')
    print(confusion)
    # ROC-AUC print 추가
    print('정확도: {0:.4f}, 정밀도: {1:.4f}, 재현율: {2:.4f}, F1： {3:.4f}, AUC：{4:.4f}'.format(accuracy, precision, recall, f1, roc_auc))

In [ ]:
from sklearn.linear_model import LogisticRegression

lr_clf = LogisticRegression(max_iter=1000)
lr_clf.fit(X_train, y_train)
lr_pred = lr_clf.predict(X_test)
lr_pred_proba = lr_clf.predict_proba(X_test)[:, 1]

# 3장에서 사용한 get_clf_eval() 함수를 이용하여 평가 수행.
get_clf_eval(y_test, lr_pred, lr_pred_proba)

앞으로 데이터 가공을 바꿔 가며 반복 평가할 것이므로, **학습·예측·평가를 한 번에 수행하는 함수**를 만든다.

In [ ]:
# 인자로 사이킷런의 Estimator객체와, 학습/테스트 데이터 세트를 입력 받아서 학습/예측/평가 수행.
def get_model_train_eval(model, ftr_train=None, ftr_test=None, tgt_train=None, tgt_test=None):
    model.fit(ftr_train, tgt_train)
    pred = model.predict(ftr_test)
    pred_proba = model.predict_proba(ftr_test)[:, 1]
    get_clf_eval(tgt_test, pred, pred_proba)

- LightGBM은 레이블 값이 극도로 불균형할 때 `boost_from_average=True`(기본값)이면 재현율과 ROC-AUC가 크게 떨어진다.
- 따라서 불균형 데이터에서는 **`boost_from_average=False`**로 설정한다.

In [ ]:
from lightgbm import LGBMClassifier

lgbm_clf = LGBMClassifier(n_estimators=1000, num_leaves=64, n_jobs=-1, boost_from_average=False)
get_model_train_eval(lgbm_clf, ftr_train=X_train, ftr_test=X_test, tgt_train=y_train, tgt_test=y_test)

가공 없이도 LightGBM이 로지스틱 회귀보다 재현율과 ROC-AUC가 높다.

### 데이터 분포도 변환 후 모델 학습/예측/평가

- 로지스틱 회귀 같은 선형 모델은 피처가 **정규 분포** 형태일 때 성능이 좋아지는 경향이 있다.
- 원래 의미가 살아 있는 `Amount` 피처의 분포를 확인한다.

In [ ]:
import seaborn as sns

plt.figure(figsize=(8, 4))
plt.xticks(range(0, 30000, 1000), rotation=60)
sns.histplot(card_df['Amount'], bins=100, kde=True)
plt.show()

- 대부분 거래 금액이 1,000불 이하이고, 26,000불까지 드물게 존재하는 **꼬리가 긴 분포**이다.
- 먼저 `StandardScaler`로 표준화해 본다. 변환한 값은 `Amount_Scaled`로 맨 앞 컬럼에 넣고 기존 `Time`, `Amount`는 삭제한다.

In [ ]:
from sklearn.preprocessing import StandardScaler
# 사이킷런의 StandardScaler를 이용하여 정규분포 형태로 Amount 피처값 변환하는 로직으로 수정.
def get_preprocessed_df(df=None):
    df_copy = df.copy()
    scaler = StandardScaler()
    amount_n = scaler.fit_transform(df_copy['Amount'].values.reshape(-1, 1))
    # 변환된 Amount를 Amount_Scaled로 피처명 변경후 DataFrame맨 앞 컬럼으로 입력
    df_copy.insert(0, 'Amount_Scaled', amount_n)
    # 기존 Time, Amount 피처 삭제
    df_copy.drop(['Time','Amount'], axis=1, inplace=True)
    return df_copy

In [ ]:
# Amount를 정규분포 형태로 변환 후 로지스틱 회귀 및 LightGBM 수행.
X_train, X_test, y_train, y_test = get_train_test_dataset(card_df)

print('### 로지스틱 회귀 예측 성능 ###')
lr_clf = LogisticRegression(max_iter=1000)
get_model_train_eval(lr_clf, ftr_train=X_train, ftr_test=X_test, tgt_train=y_train, tgt_test=y_test)

print('### LightGBM 예측 성능 ###')
lgbm_clf = LGBMClassifier(n_estimators=1000, num_leaves=64, n_jobs=-1, boost_from_average=False)
get_model_train_eval(lgbm_clf, ftr_train=X_train, ftr_test=X_test, tgt_train=y_train, tgt_test=y_test)

- 표준화 후에는 두 모델 모두 성능이 크게 달라지지 않는다.
- 이번에는 꼬리가 긴 분포를 정규 분포에 가깝게 만들 때 자주 쓰는 **로그 변환**을 적용한다. `np.log1p()`는 `log(1+x)`를 계산해 0 값에서도 문제가 생기지 않는다.
- 로그 변환은 원래 값을 상대적으로 작은 범위로 줄여 **왜곡된 분포를 완화**하는 데 효과적이다.

In [ ]:
def get_preprocessed_df(df=None):
    df_copy = df.copy()
    # 넘파이의 log1p( )를 이용하여 Amount를 로그 변환
    amount_n = np.log1p(df_copy['Amount'])
    df_copy.insert(0, 'Amount_Scaled', amount_n)
    df_copy.drop(['Time','Amount'], axis=1, inplace=True)
    return df_copy

In [ ]:
X_train, X_test, y_train, y_test = get_train_test_dataset(card_df)

print('### 로지스틱 회귀 예측 성능 ###')
get_model_train_eval(lr_clf, ftr_train=X_train, ftr_test=X_test, tgt_train=y_train, tgt_test=y_test)

print('### LightGBM 예측 성능 ###')
get_model_train_eval(lgbm_clf, ftr_train=X_train, ftr_test=X_test, tgt_train=y_train, tgt_test=y_test)

로그 변환 후 로지스틱 회귀의 정밀도·재현율이 약간 개선되고, LightGBM은 재현율이 조금 오른다.

### 이상치 데이터 제거 후 모델 학습/예측/평가

**이상치(Outlier)**: 전체 데이터의 패턴에서 벗어난 극단적인 값. 이상치 때문에 머신러닝 모델의 성능이 떨어지는 경우가 많다.

**IQR(Inter Quantile Range) 방식**
- IQR = 3/4 분위수(Q3) - 1/4 분위수(Q1)
- 박스 플롯에서 박스 길이에 해당하며, 보통 IQR에 **1.5를 곱한 값**을 기준으로 한다.
- **Q1 - 1.5×IQR 보다 작거나, Q3 + 1.5×IQR 보다 큰 값**을 이상치로 본다.
- 1.5는 상황에 따라 바꿀 수 있다(`weight` 인자).

**어떤 피처의 이상치를 제거할까?**
- 피처가 많으면 모든 피처의 이상치를 지우는 것은 시간이 오래 걸리고 효과도 없다.
- **결정 레이블(`Class`)과 상관성이 높은 피처**를 골라 이상치를 제거하는 것이 좋다.
- 상관관계 히트맵에서 `Class`와 음의 상관관계가 가장 높은 피처는 `V14`, `V17`이며, 이 중 `V14`의 이상치를 제거한다.

In [ ]:
import seaborn as sns

plt.figure(figsize=(9, 9))
corr = card_df.corr()
sns.heatmap(corr, cmap='RdBu')

`get_outlier()`는 **사기(`Class=1`) 데이터만** 대상으로 IQR을 구해 이상치의 인덱스를 반환한다.

In [ ]:
import numpy as np

def get_outlier(df=None, column=None, weight=1.5):
    # fraud에 해당하는 column 데이터만 추출, 1/4 분위와 3/4 분위 지점을 np.percentile로 구함.
    fraud = df[df['Class']==1][column]
    quantile_25 = np.percentile(fraud.values, 25)
    quantile_75 = np.percentile(fraud.values, 75)
    # IQR을 구하고, IQR에 1.5를 곱하여 최대값과 최소값 지점 구함.
    iqr = quantile_75 - quantile_25
    iqr_weight = iqr * weight
    lowest_val = quantile_25 - iqr_weight
    highest_val = quantile_75 + iqr_weight
    # 최대값 보다 크거나, 최소값 보다 작은 값을 아웃라이어로 설정하고 DataFrame index 반환.
    outlier_index = fraud[(fraud < lowest_val) | (fraud > highest_val)].index
    return outlier_index

In [ ]:
outlier_index = get_outlier(df=card_df, column='V14', weight=1.5)
print('이상치 데이터 인덱스:', outlier_index)

총 4개의 이상치가 발견된다. 로그 변환에 이어 이 이상치를 삭제하도록 `get_preprocessed_df()`를 바꾸고 다시 평가한다.

In [ ]:
# get_processed_df( )를 로그 변환 후 V14 피처의 이상치 데이터를 삭제하는 로직으로 변경.
def get_preprocessed_df(df=None):
    df_copy = df.copy()
    amount_n = np.log1p(df_copy['Amount'])
    df_copy.insert(0, 'Amount_Scaled', amount_n)
    df_copy.drop(['Time','Amount'], axis=1, inplace=True)
    # 이상치 데이터 삭제하는 로직 추가
    outlier_index = get_outlier(df=df_copy, column='V14', weight=1.5)
    df_copy.drop(outlier_index, axis=0, inplace=True)
    return df_copy

X_train, X_test, y_train, y_test = get_train_test_dataset(card_df)
print('### 로지스틱 회귀 예측 성능 ###')
get_model_train_eval(lr_clf, ftr_train=X_train, ftr_test=X_test, tgt_train=y_train, tgt_test=y_test)
print('### LightGBM 예측 성능 ###')
get_model_train_eval(lgbm_clf, ftr_train=X_train, ftr_test=X_test, tgt_train=y_train, tgt_test=y_test)

이상치 4개만 지웠는데도 두 모델 모두 **재현율이 크게 향상**된다.

### SMOTE 오버 샘플링 적용 후 모델 학습/예측/평가

- SMOTE는 반드시 **학습 데이터에만** 적용해야 한다. 검증·테스트 데이터까지 증식하면 원본 데이터가 아닌 데이터로 평가하게 되어 올바른 평가가 되지 않는다.
- `imbalanced-learn`의 `SMOTE.fit_resample()`로 증식한다.

In [ ]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=0)
X_train_over, y_train_over = smote.fit_resample(X_train, y_train)
print('SMOTE 적용 전 학습용 피처/레이블 데이터 세트: ', X_train.shape, y_train.shape)
print('SMOTE 적용 후 학습용 피처/레이블 데이터 세트: ', X_train_over.shape, y_train_over.shape)
print('SMOTE 적용 후 레이블 값 분포: \n', pd.Series(y_train_over).value_counts())

SMOTE 적용 후 학습 데이터가 약 2배로 늘어나고, 0과 1 레이블 개수가 같아졌다. 먼저 로지스틱 회귀로 학습한다.

In [ ]:
lr_clf = LogisticRegression(max_iter=1000)
# ftr_train과 tgt_train 인자값이 SMOTE 증식된 X_train_over와 y_train_over로 변경됨에 유의
get_model_train_eval(lr_clf, ftr_train=X_train_over, ftr_test=X_test, tgt_train=y_train_over, tgt_test=y_test)

- 재현율은 크게 올랐지만 **정밀도가 매우 낮아졌다.**
- 오버 샘플링으로 사기(1) 데이터를 너무 많이 학습해서, 실제 테스트 데이터에서 정상 거래까지 사기로 지나치게 많이 예측했기 때문이다.
- 분류 결정 임곗값에 따라 정밀도와 재현율이 어떻게 바뀌는지 곡선으로 확인한다.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from sklearn.metrics import precision_recall_curve
%matplotlib inline

def precision_recall_curve_plot(y_test , pred_proba_c1):
    # threshold ndarray와 이 threshold에 따른 정밀도, 재현율 ndarray 추출.
    precisions, recalls, thresholds = precision_recall_curve( y_test, pred_proba_c1)

    # X축을 threshold값으로, Y축은 정밀도, 재현율 값으로 각각 Plot 수행. 정밀도는 점선으로 표시
    plt.figure(figsize=(8,6))
    threshold_boundary = thresholds.shape[0]
    plt.plot(thresholds, precisions[0:threshold_boundary], linestyle='--', label='precision')
    plt.plot(thresholds, recalls[0:threshold_boundary],label='recall')

    # threshold 값 X 축의 Scale을 0.1 단위로 변경
    start, end = plt.xlim()
    plt.xticks(np.round(np.arange(start, end, 0.1),2))

    # x축, y축 label과 legend, 그리고 grid 설정
    plt.xlabel('Threshold value'); plt.ylabel('Precision and Recall value')
    plt.legend(); plt.grid()
    plt.show()

In [ ]:
precision_recall_curve_plot(y_test, lr_clf.predict_proba(X_test)[:, 1])

- 임곗값이 0.99 이하에서는 재현율이 매우 높고 정밀도가 극히 낮다가, 0.99 이상에서야 정밀도가 급격히 올라간다.
- 곡선이 너무 급격하게 변해서 **임곗값 조정으로 적절한 지점을 찾기 어렵다.** 즉, SMOTE 후 로지스틱 회귀는 실제 적용하기에 부적절하다.
- 이번에는 LightGBM에 SMOTE 데이터를 적용한다.

In [ ]:
lgbm_clf = LGBMClassifier(n_estimators=1000, num_leaves=64, n_jobs=-1, boost_from_average=False)
get_model_train_eval(lgbm_clf, ftr_train=X_train_over, ftr_test=X_test,
                     tgt_train=y_train_over, tgt_test=y_test)

- LightGBM은 SMOTE 적용 후 재현율이 이상치 제거만 했을 때보다 높아지고, 정밀도는 그보다 낮아진다.
- SMOTE를 적용하면 일반적으로 **재현율은 높아지고 정밀도는 낮아진다.** 따라서 정밀도 하락을 최소화하면서 재현율을 높이는 것이 좋은 SMOTE 적용의 핵심이다.

## 정리

| 주제 | 핵심 |
|---|---|
| 산탄데르 고객 만족 | 불균형 데이터라 ROC-AUC로 평가한다. -999999 같은 예외값은 최빈값으로 대체하고 ID는 삭제한다 |
| 조기 중단용 검증 세트 | 테스트 데이터 대신 학습 데이터를 다시 나눈 검증 세트를 쓴다 |
| HyperOpt 튜닝 | KFold 교차 검증 평균 ROC-AUC에 -1을 곱해 반환한다. XGBoost와 LightGBM 성능은 비슷하지만 LightGBM이 훨씬 빠르다 |
| LightGBM 버전 차이 | 4.x부터는 `early_stopping_rounds` 대신 `callbacks=[early_stopping(n)]`을 쓴다 |
| 불균형 데이터 + LightGBM | `boost_from_average=False`로 설정해야 재현율과 ROC-AUC가 떨어지지 않는다 |
| 분포 변환 | 꼬리가 긴 피처는 `np.log1p()` 로그 변환이 효과적이다 |
| 이상치 제거 | IQR × 1.5 기준. 레이블과 상관성이 높은 피처(V14)의 이상치만 지워도 재현율이 크게 오른다 |
| SMOTE | 학습 데이터에만 적용한다. 재현율은 오르고 정밀도는 떨어지는 경향이 있다 |